In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

##############################################################
# 1. LOAD DATASET
##############################################################

data = pd.read_csv("pos_tags.csv")

print(data.head())

##############################################################
# 2. PREPROCESS DATA
##############################################################

sentences = []

for sid, group in data.groupby("sentence_id"):
    words = list(group["word"])
    tags = list(group["tag"])
    sentences.append((words, tags))

print("Total Sentences :", len(sentences))

##############################################################
# 3. TRAIN TEST SPLIT
##############################################################

train_sentences, test_sentences = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print("Training :", len(train_sentences))
print("Testing :", len(test_sentences))

##############################################################
# 4. VOCABULARY AND TAG SET
##############################################################

vocab = set()
tagset = set()

for words, tags in train_sentences:
    vocab.update(words)
    tagset.update(tags)

vocab = sorted(vocab)
tagset = sorted(tagset)

word_to_index = {w:i for i,w in enumerate(vocab)}
tag_to_index = {t:i for i,t in enumerate(tagset)}

index_to_tag = {i:t for t,i in tag_to_index.items()}

num_tags = len(tagset)
num_words = len(vocab)

print("Vocabulary :", num_words)
print("Tags :", num_tags)

##############################################################
# 5. BUILD COUNT MATRICES
##############################################################

initial_counts = np.ones(num_tags)

transition_counts = np.ones((num_tags, num_tags))

emission_counts = np.ones((num_tags, num_words))

for words, tags in train_sentences:

    first_tag = tags[0]
    initial_counts[tag_to_index[first_tag]] += 1

    for i in range(len(tags)):

        tag_idx = tag_to_index[tags[i]]

        if words[i] in word_to_index:
            word_idx = word_to_index[words[i]]
            emission_counts[tag_idx, word_idx] += 1

        if i > 0:
            prev = tag_to_index[tags[i-1]]
            curr = tag_idx
            transition_counts[prev, curr] += 1

##############################################################
# 6. PROBABILITY MATRICES
##############################################################

initial_prob = initial_counts / initial_counts.sum()

transition_prob = transition_counts / transition_counts.sum(axis=1, keepdims=True)

emission_prob = emission_counts / emission_counts.sum(axis=1, keepdims=True)

##############################################################
# 7. CONVERT TO LOG SPACE
##############################################################

log_initial = np.log(initial_prob)

log_transition = np.log(transition_prob)

log_emission = np.log(emission_prob)

##############################################################
# UNKNOWN WORD PROBABILITY
##############################################################

unknown_log_prob = np.log(1 / (num_words + 1))

##############################################################
# 8. VITERBI ALGORITHM (VECTORIZED)
##############################################################

def viterbi(sentence):

    T = len(sentence)

    dp = np.full((num_tags, T), -np.inf)

    backpointer = np.zeros((num_tags, T), dtype=int)

    # First word

    if sentence[0] in word_to_index:
        emit = log_emission[:, word_to_index[sentence[0]]]
    else:
        emit = np.full(num_tags, unknown_log_prob)

    dp[:,0] = log_initial + emit

    # Remaining words

    for t in range(1, T):

        if sentence[t] in word_to_index:
            emit = log_emission[:, word_to_index[sentence[t]]]
        else:
            emit = np.full(num_tags, unknown_log_prob)

        scores = dp[:,t-1][:,None] + log_transition

        backpointer[:,t] = np.argmax(scores, axis=0)

        dp[:,t] = np.max(scores, axis=0) + emit

    best_last = np.argmax(dp[:,T-1])

    best_path = [best_last]

    for t in range(T-1,0,-1):
        best_last = backpointer[best_last,t]
        best_path.append(best_last)

    best_path.reverse()

    return [index_to_tag[i] for i in best_path]

##############################################################
# 9. TEST ON TEST SET
##############################################################

true_tags = []
pred_tags = []

for words, tags in test_sentences:

    predicted = viterbi(words)

    true_tags.extend(tags)
    pred_tags.extend(predicted)

##############################################################
# 10. EVALUATION
##############################################################

accuracy = accuracy_score(true_tags, pred_tags)

print("\nAccuracy :", accuracy)

print("\nClassification Report\n")

print(classification_report(
    true_tags,
    pred_tags,
    zero_division=0
))

##############################################################
# 11. TEST ON UNSEEN SENTENCES
##############################################################

test_examples = [

    "The cat sat on the mat",

    "Artificial intelligence is changing education",

    "She enjoys reading books",

    "OpenAI develops powerful language models",

    "Students are learning machine learning"

]

print("\n=============================")
print("UNSEEN SENTENCE PREDICTIONS")
print("=============================\n")

for sentence in test_examples:

    words = sentence.split()

    tags = viterbi(words)

    print("\nSentence:")
    print(sentence)

    print("\nPredicted Tags:")

    for w, t in zip(words, tags):
        print(f"{w:15} {t}")

   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG
Total Sentences : 370100
Training : 296080
Testing : 74020
Vocabulary : 296080
Tags : 24

Accuracy : 0.6233450418805728

Classification Report

              precision    recall  f1-score   support

          CC       0.00      0.00      0.00         3
          CD       0.00      0.00      0.00         1
          DT       0.00      0.00      0.00         6
          IN       0.00      0.00      0.00        12
          JJ       0.00      0.00      0.00      7847
         JJR       0.00      0.00      0.00         2
         JJS       0.00      0.00      0.00        62
          MD       0.00      0.00      0.00         3
          NN       0.62      1.00      0.77     46140
         NNS       0.00      0.00      0.00      9682
         PRP       0.00      0.00      0.00         4
        PRP$       0.00      0.00      